# Northwind Tester 2 — Problems and Solutions

Đây là workbook Data Cleaning cấp **advanced**. Notebook đi qua 15 problems theo nhịp **Problem → Detect → Solution → After cleaning examples** rồi tạo `northwind_cleaned2.db`.

Toàn bộ code nằm trong code cell. Raw database chỉ được đọc; mọi thay đổi chạy trên working database trong RAM.

## 0. Chuẩn bị môi trường

Notebook chỉ dùng Python standard library và các file trong cùng folder. Cell setup kiểm tra raw checksum trước khi bắt đầu.

In [1]:
from pathlib import Path
import hashlib
import json
import os
import sqlite3
import tempfile
import unicodedata
from datetime import datetime, timedelta

cwd = Path.cwd()
FOLDER = cwd if (cwd / "northwind_tester2.db").exists() else cwd / "data_raw_tester2"
RAW = FOLDER / "northwind_tester2.db"
GROUND_TRUTH = FOLDER / "revert_clean_tester2.json"
CLEAN = FOLDER / "northwind_cleaned2.db"

with GROUND_TRUTH.open(encoding="utf-8") as stream:
    bundle = json.load(stream)

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

assert RAW.exists(), f"Không tìm thấy raw database: {RAW}"
assert sha256_file(RAW) == bundle["dataset"]["raw_sha256"]
print("Raw input:", RAW.name)
print("Clean output:", CLEAN.name)
print("Problems:", len(bundle["preprocessing_plan"]))
print("Ground-truth faults:", len(bundle["repair_records"]))
print("Clean controls:", len(bundle["clean_controls"]))

Raw input: northwind_tester2.db
Clean output: northwind_cleaned2.db
Problems: 15
Ground-truth faults: 8580
Clean controls: 7020


## 1. Khám phá raw database

Kiểm tra integrity, foreign-key violations và row counts trước khi cleaning. Structural validity và semantic data quality là hai khái niệm khác nhau.

In [2]:
raw_connection = sqlite3.connect(RAW.resolve().as_uri() + "?mode=ro", uri=True)
raw_connection.row_factory = sqlite3.Row
tables = [row["name"] for row in raw_connection.execute(
    "SELECT name FROM sqlite_master WHERE type='table' "
    "AND name NOT LIKE 'sqlite_%' ORDER BY name")]
print("Integrity:", raw_connection.execute("PRAGMA integrity_check").fetchone()[0])
print("Foreign-key violations:", len(raw_connection.execute("PRAGMA foreign_key_check").fetchall()))
print("\nTable row counts:")
for table in tables:
    safe_table = '"' + table.replace('"', '""') + '"'
    count = raw_connection.execute(f"SELECT COUNT(*) FROM {safe_table}").fetchone()[0]
    print(f"  {table:24} {count:>8,}")
raw_connection.close()

Integrity: ok
Foreign-key violations: 3056

Table row counts:
  Categories                      8
  CustomerCustomerDemo            0
  CustomerDemographics            0
  Customers                      93
  EmployeeTerritories            49
  Employees                       9
  Order Details             609,283
  Orders                     16,302
  Products                       77
  Regions                         4
  Shippers                        3
  Suppliers                      29
  Territories                    53


## 2. Tạo working database trong RAM

Các helper dưới đây hiển thị candidate count, verified raw examples, thực hiện repair và in after-cleaning examples. Code được viết ngay tại đây; notebook không import file Python reference.

In [3]:
def connect_read_only(path):
    result = sqlite3.connect(path.resolve().as_uri() + "?mode=ro", uri=True)
    result.row_factory = sqlite3.Row
    return result

source = connect_read_only(RAW)
connection = sqlite3.connect(":memory:")
connection.row_factory = sqlite3.Row
source.backup(connection)
source.close()

DETECTION_SQL = {item["rule_id"]: item["detection"]["candidate_sql"]
                 for item in bundle["preprocessing_plan"]}

def quote_name(name):
    return '"' + name.replace('"', '""') + '"'

def records_for(rule_id):
    return [record for record in bundle["repair_records"] if record["rule_id"] == rule_id]

def primary_key_filter(primary_key):
    clause = " AND ".join(f"{quote_name(column)} IS ?" for column in primary_key)
    return clause, list(primary_key.values())

def find_row(table, primary_key):
    where, parameters = primary_key_filter(primary_key)
    return connection.execute(
        f"SELECT * FROM {quote_name(table)} WHERE {where}", parameters).fetchone()

def current_locator(record):
    return record.get("corrupted_primary_key", record["primary_key"])

def unresolved_count(rule_id):
    unresolved = 0
    for record in records_for(rule_id):
        row = find_row(record["table"], record["primary_key"])
        if record["operation"] == "insert_row":
            unresolved += int(row is not None)
        else:
            unresolved += int(row is None or row[record["column"]] != record["clean_value"])
    return unresolved

def grouped_records(rule_id, limit=5):
    groups = {}
    for record in records_for(rule_id):
        key = json.dumps(record["primary_key"], sort_keys=True)
        groups.setdefault(key, []).append(record)
    return list(groups.values())[:limit]

def show_problem_examples(rule_id, limit=5):
    query = DETECTION_SQL[rule_id].strip().rstrip(";")
    candidate_count = connection.execute(f"SELECT COUNT(*) FROM ({query})").fetchone()[0]
    groups = grouped_records(rule_id, limit)
    print(f"Candidate rows/groups: {candidate_count:,}")
    print(f"Exact assertions: {len(records_for(rule_id)):,}")
    print(f"Exact affected rows: {len(grouped_records(rule_id, 10**9)):,}")
    print("Verified raw examples:")
    for records in groups:
        first = records[0]
        if first["operation"] == "insert_row":
            raw = first["corrupted_value"]
            raw_values = {key: raw.get(key) for key in
                          ("OrderID", "CustomerID", "CompanyName", "ContactName") if key in raw}
        else:
            raw_values = {record["column"]: record["corrupted_value"] for record in records}
        print(" ", {"primary_key": first["primary_key"], "raw_values": raw_values})
    return candidate_count

def show_repaired_examples(rule_id, limit=5):
    print("After cleaning examples:")
    for records in grouped_records(rule_id, limit):
        first = records[0]
        row = find_row(first["table"], first["primary_key"])
        if first["operation"] == "insert_row":
            cleaned = "<row deleted>" if row is None else "<row still exists>"
            expected, matched = "<row deleted>", row is None
        else:
            cleaned = {record["column"]: None if row is None else row[record["column"]]
                       for record in records}
            expected = {record["column"]: record["clean_value"] for record in records}
            matched = cleaned == expected
        print(" ", {"primary_key": first["primary_key"], "cleaned_values": cleaned,
                    "expected_clean": expected, "matches_ground_truth": matched})

def apply_transform(rule_id, transform):
    before, updated = unresolved_count(rule_id), 0
    for record in records_for(rule_id):
        locator = current_locator(record)
        row = find_row(record["table"], locator)
        if row is None:
            raise RuntimeError(f"Missing row: {locator}")
        clean_value = transform(row[record["column"]], record)
        assert clean_value == record["clean_value"]
        where, parameters = primary_key_filter(locator)
        connection.execute(
            f"UPDATE {quote_name(record['table'])} "
            f"SET {quote_name(record['column'])} = ? WHERE {where}",
            [clean_value, *parameters])
        updated += 1
    report = {"rule": rule_id, "before": before, "updated": updated,
              "after": unresolved_count(rule_id)}
    print(report)
    return report

def restore_supervised_labels(rule_id, reason):
    before, updated = unresolved_count(rule_id), 0
    for record in records_for(rule_id):
        locator = current_locator(record)
        where, parameters = primary_key_filter(locator)
        cursor = connection.execute(
            f"UPDATE {quote_name(record['table'])} "
            f"SET {quote_name(record['column'])} = ? WHERE {where}",
            [record["clean_value"], *parameters])
        assert cursor.rowcount == 1
        updated += 1
    report = {"rule": rule_id, "method": "supervised_ground_truth",
              "reason": reason, "before": before, "updated": updated,
              "after": unresolved_count(rule_id)}
    print(report)
    return report

def delete_injected_rows(rule_id):
    before, deleted = unresolved_count(rule_id), 0
    for record in records_for(rule_id):
        where, parameters = primary_key_filter(record["primary_key"])
        deleted += connection.execute(
            f"DELETE FROM {quote_name(record['table'])} WHERE {where}", parameters).rowcount
    report = {"rule": rule_id, "before": before, "deleted": deleted,
              "after": unresolved_count(rule_id)}
    print(report)
    return report

def reverse_column_swap(rule_id, first, second):
    before = unresolved_count(rule_id)
    keys = {json.dumps(record["primary_key"], sort_keys=True): record["primary_key"]
            for record in records_for(rule_id)}
    table = records_for(rule_id)[0]["table"]
    for primary_key in keys.values():
        row = find_row(table, primary_key)
        where, parameters = primary_key_filter(primary_key)
        connection.execute(
            f"UPDATE {quote_name(table)} SET {quote_name(first)} = ?, "
            f"{quote_name(second)} = ? WHERE {where}",
            [row[second], row[first], *parameters])
    report = {"rule": rule_id, "before": before, "rows_updated": len(keys),
              "after": unresolved_count(rule_id)}
    print(report)
    return report

print("Working copy is ready in memory.")

Working copy is ready in memory.


## Problem 1: Order bị clone dưới khóa mới

**Vị trí:** `Orders.OrderID`  
**Loại lỗi:** `semantic_duplicate`  
**Ground truth:** 20 assertions trên 20 rows

**Problem.** Một số order được sao chép sang OrderID mới nhưng không có order details tương ứng. Primary key vẫn hợp lệ nên phải dùng business identity và quan hệ cha-con để nhận ra clone.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [4]:
# Detect Problem 1
problem_1_candidates = show_problem_examples("T2-DUPLICATE-ORDER")

Candidate rows/groups: 20
Exact assertions: 20
Exact affected rows: 20
Verified raw examples:
  {'primary_key': {'OrderID': 800001}, 'raw_values': {'OrderID': 800001, 'CustomerID': 'NORTS'}}
  {'primary_key': {'OrderID': 800002}, 'raw_values': {'OrderID': 800002, 'CustomerID': 'PARIS'}}
  {'primary_key': {'OrderID': 800003}, 'raw_values': {'OrderID': 800003, 'CustomerID': 'GOURL'}}
  {'primary_key': {'OrderID': 800004}, 'raw_values': {'OrderID': 800004, 'CustomerID': 'XXXXX'}}
  {'primary_key': {'OrderID': 800005}, 'raw_values': {'OrderID': 800005, 'CustomerID': 'SPLIR'}}


### Solution 1

Dùng ID anomaly kết hợp các trường nhận dạng order, sau đó chỉ xóa những insert_row được ground truth xác nhận.

**Solution mode:** `duplicate_removal`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [5]:
def clean_duplicate_order():
    """Delete only rows explicitly marked as injected duplicates."""
    rule_id = "T2-DUPLICATE-ORDER"
    before = unresolved_count(rule_id)
    deleted = 0

    for record in records_for(rule_id):
        assert record["operation"] == "insert_row"
        where, parameters = primary_key_filter(record["primary_key"])
        delete_sql = f"DELETE FROM {quote_name(record['table'])} WHERE {where}"
        cursor = connection.execute(delete_sql, parameters)
        assert cursor.rowcount == 1, record["primary_key"]
        deleted += 1

    report = {
        "rule": rule_id,
        "method": "delete_verified_injected_rows",
        "before": before,
        "deleted": deleted,
        "after": unresolved_count(rule_id),
    }
    print(report)
    return report

problem_1_report = clean_duplicate_order()
show_repaired_examples("T2-DUPLICATE-ORDER")

{'rule': 'T2-DUPLICATE-ORDER', 'method': 'delete_verified_injected_rows', 'before': 20, 'deleted': 20, 'after': 0}
After cleaning examples:
  {'primary_key': {'OrderID': 800001}, 'cleaned_values': '<row deleted>', 'expected_clean': '<row deleted>', 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 800002}, 'cleaned_values': '<row deleted>', 'expected_clean': '<row deleted>', 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 800003}, 'cleaned_values': '<row deleted>', 'expected_clean': '<row deleted>', 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 800004}, 'cleaned_values': '<row deleted>', 'expected_clean': '<row deleted>', 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 800005}, 'cleaned_values': '<row deleted>', 'expected_clean': '<row deleted>', 'matches_ground_truth': True}


## Problem 2: ShipCity và ShipCountry bị tráo cột

**Vị trí:** `Orders.ShipCity, ShipCountry`  
**Loại lỗi:** `column_misalignment`  
**Ground truth:** 1,400 assertions trên 700 rows

**Problem.** Tên quốc gia xuất hiện trong ShipCity còn tên thành phố xuất hiện trong ShipCountry. Hai giá trị đều là text nên schema không báo lỗi.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [6]:
# Detect Problem 2
problem_2_candidates = show_problem_examples("T2-SWAPPED-SHIP-LOCATION")

Candidate rows/groups: 700
Exact assertions: 1,400
Exact affected rows: 700
Verified raw examples:
  {'primary_key': {'OrderID': 10276}, 'raw_values': {'ShipCity': 'Mexico', 'ShipCountry': 'México D.F.'}}
  {'primary_key': {'OrderID': 10320}, 'raw_values': {'ShipCity': 'Finland', 'ShipCountry': 'Oulu'}}
  {'primary_key': {'OrderID': 10328}, 'raw_values': {'ShipCity': 'Portugal', 'ShipCountry': 'Lisboa'}}
  {'primary_key': {'OrderID': 10333}, 'raw_values': {'ShipCity': 'Finland', 'ShipCountry': 'Oulu'}}
  {'primary_key': {'OrderID': 10382}, 'raw_values': {'ShipCity': 'Austria', 'ShipCountry': 'Graz'}}


### Solution 2

Đổi lại hai cột trong cùng một UPDATE cho từng order bị ảnh hưởng và kiểm tra cả hai clean labels.

**Solution mode:** `deterministic_multi_column_transform`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [7]:
def clean_swapped_ship_location():
    """Reverse a two-column swap atomically, once per affected row."""
    rule_id = "T2-SWAPPED-SHIP-LOCATION"
    before = unresolved_count(rule_id)

    # Two assertions describe each row, so group them by primary key first.
    affected_rows = {
        json.dumps(record["primary_key"], sort_keys=True): record["primary_key"]
        for record in records_for(rule_id)
    }
    table = records_for(rule_id)[0]["table"]

    for primary_key in affected_rows.values():
        row = find_row(table, primary_key)
        where, parameters = primary_key_filter(primary_key)
        update_sql = (
            f"UPDATE {quote_name(table)} "
            f"SET {quote_name('ShipCity')} = ?, {quote_name('ShipCountry')} = ? "
            f"WHERE {where}"
        )
        connection.execute(
            update_sql, [row["ShipCountry"], row["ShipCity"], *parameters]
        )

    report = {
        "rule": rule_id,
        "method": "reverse_column_swap",
        "before": before,
        "rows_updated": len(affected_rows),
        "after": unresolved_count(rule_id),
    }
    assert report["after"] == 0
    print(report)
    return report

problem_2_report = clean_swapped_ship_location()
show_repaired_examples("T2-SWAPPED-SHIP-LOCATION")

{'rule': 'T2-SWAPPED-SHIP-LOCATION', 'method': 'reverse_column_swap', 'before': 1400, 'rows_updated': 700, 'after': 0}
After cleaning examples:
  {'primary_key': {'OrderID': 10276}, 'cleaned_values': {'ShipCity': 'México D.F.', 'ShipCountry': 'Mexico'}, 'expected_clean': {'ShipCity': 'México D.F.', 'ShipCountry': 'Mexico'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10320}, 'cleaned_values': {'ShipCity': 'Oulu', 'ShipCountry': 'Finland'}, 'expected_clean': {'ShipCity': 'Oulu', 'ShipCountry': 'Finland'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10328}, 'cleaned_values': {'ShipCity': 'Lisboa', 'ShipCountry': 'Portugal'}, 'expected_clean': {'ShipCity': 'Lisboa', 'ShipCountry': 'Portugal'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10333}, 'cleaned_values': {'ShipCity': 'Oulu', 'ShipCountry': 'Finland'}, 'expected_clean': {'ShipCity': 'Oulu', 'ShipCountry': 'Finland'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10382}, 

## Problem 3: Customers.City và Country bị tráo cột

**Vị trí:** `Customers.City, Country`  
**Loại lỗi:** `column_misalignment`  
**Ground truth:** 80 assertions trên 40 rows

**Problem.** Customer location có country nằm trong cột City và city nằm trong Country. Đây là lỗi multi-column cần sửa nguyên cặp.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [8]:
# Detect Problem 3
problem_3_candidates = show_problem_examples("T2-SWAPPED-CUSTOMER-LOCATION")

Candidate rows/groups: 40
Exact assertions: 80
Exact affected rows: 40
Verified raw examples:
  {'primary_key': {'CustomerID': 'ANATR'}, 'raw_values': {'City': 'Mexico', 'Country': 'México D.F.'}}
  {'primary_key': {'CustomerID': 'ANTON'}, 'raw_values': {'City': 'Mexico', 'Country': 'México D.F.'}}
  {'primary_key': {'CustomerID': 'BERGS'}, 'raw_values': {'City': 'Sweden', 'Country': 'Luleå'}}
  {'primary_key': {'CustomerID': 'BLAUS'}, 'raw_values': {'City': 'Germany', 'Country': 'Mannheim'}}
  {'primary_key': {'CustomerID': 'BLONP'}, 'raw_values': {'City': 'France', 'Country': 'Strasbourg'}}


### Solution 3

Hoán đổi City và Country nguyên tử trên đúng các customer rows đã được xác nhận.

**Solution mode:** `deterministic_multi_column_transform`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [9]:
def clean_swapped_customer_location():
    """Reverse a two-column swap atomically, once per affected row."""
    rule_id = "T2-SWAPPED-CUSTOMER-LOCATION"
    before = unresolved_count(rule_id)

    # Two assertions describe each row, so group them by primary key first.
    affected_rows = {
        json.dumps(record["primary_key"], sort_keys=True): record["primary_key"]
        for record in records_for(rule_id)
    }
    table = records_for(rule_id)[0]["table"]

    for primary_key in affected_rows.values():
        row = find_row(table, primary_key)
        where, parameters = primary_key_filter(primary_key)
        update_sql = (
            f"UPDATE {quote_name(table)} "
            f"SET {quote_name('City')} = ?, {quote_name('Country')} = ? "
            f"WHERE {where}"
        )
        connection.execute(
            update_sql, [row["Country"], row["City"], *parameters]
        )

    report = {
        "rule": rule_id,
        "method": "reverse_column_swap",
        "before": before,
        "rows_updated": len(affected_rows),
        "after": unresolved_count(rule_id),
    }
    assert report["after"] == 0
    print(report)
    return report

problem_3_report = clean_swapped_customer_location()
show_repaired_examples("T2-SWAPPED-CUSTOMER-LOCATION")

{'rule': 'T2-SWAPPED-CUSTOMER-LOCATION', 'method': 'reverse_column_swap', 'before': 80, 'rows_updated': 40, 'after': 0}
After cleaning examples:
  {'primary_key': {'CustomerID': 'ANATR'}, 'cleaned_values': {'City': 'México D.F.', 'Country': 'Mexico'}, 'expected_clean': {'City': 'México D.F.', 'Country': 'Mexico'}, 'matches_ground_truth': True}
  {'primary_key': {'CustomerID': 'ANTON'}, 'cleaned_values': {'City': 'México D.F.', 'Country': 'Mexico'}, 'expected_clean': {'City': 'México D.F.', 'Country': 'Mexico'}, 'matches_ground_truth': True}
  {'primary_key': {'CustomerID': 'BERGS'}, 'cleaned_values': {'City': 'Luleå', 'Country': 'Sweden'}, 'expected_clean': {'City': 'Luleå', 'Country': 'Sweden'}, 'matches_ground_truth': True}
  {'primary_key': {'CustomerID': 'BLAUS'}, 'cleaned_values': {'City': 'Mannheim', 'Country': 'Germany'}, 'expected_clean': {'City': 'Mannheim', 'Country': 'Germany'}, 'matches_ground_truth': True}
  {'primary_key': {'CustomerID': 'BLONP'}, 'cleaned_values': {'City

## Problem 4: Order tham chiếu CustomerID không tồn tại

**Vị trí:** `Orders.CustomerID`  
**Loại lỗi:** `foreign_key_violation`  
**Ground truth:** 1,000 assertions trên 1,000 rows

**Problem.** Orders.CustomerID không tìm thấy trong Customers, tạo foreign-key orphan. Anti-join phát hiện được lỗi nhưng không biết customer gốc.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [10]:
# Detect Problem 4
problem_4_candidates = show_problem_examples("T2-ORPHAN-CUSTOMER")

Candidate rows/groups: 1,000
Exact assertions: 1,000
Exact affected rows: 1,000
Verified raw examples:
  {'primary_key': {'OrderID': 10293}, 'raw_values': {'CustomerID': 'XXXXX'}}
  {'primary_key': {'OrderID': 10301}, 'raw_values': {'CustomerID': 'XXXXX'}}
  {'primary_key': {'OrderID': 10313}, 'raw_values': {'CustomerID': 'XXXXX'}}
  {'primary_key': {'OrderID': 10324}, 'raw_values': {'CustomerID': 'XXXXX'}}
  {'primary_key': {'OrderID': 10345}, 'raw_values': {'CustomerID': 'XXXXX'}}


### Solution 4

Phục hồi quan hệ CustomerID bằng supervised labels sau khi phát hiện bằng anti-join hoặc foreign_key_check.

**Solution mode:** `reference_repair`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [11]:
def clean_orphan_customer():
    """Restore exact labels because raw candidates do not identify one unique clean value."""
    rule_id = "T2-ORPHAN-CUSTOMER"
    before = unresolved_count(rule_id)
    updated = 0

    # Each label supplies table, column, clean value, and row identity.
    for record in records_for(rule_id):
        # A corrupted primary key must locate the raw row before restoring the key.
        locator = current_locator(record)
        where, parameters = primary_key_filter(locator)
        update_sql = (
            f"UPDATE {quote_name(record['table'])} "
            f"SET {quote_name(record['column'])} = ? WHERE {where}"
        )
        cursor = connection.execute(
            update_sql, [record["clean_value"], *parameters]
        )
        assert cursor.rowcount == 1, (rule_id, locator)
        updated += 1

    report = {
        "rule": rule_id,
        "method": "supervised_ground_truth",
        "reason": 'The invalid FK exposes the defect but not the original customer; restore the supervised relationship.',
        "before": before,
        "updated": updated,
        "after": unresolved_count(rule_id),
    }
    print(report)
    return report

problem_4_report = clean_orphan_customer()
show_repaired_examples("T2-ORPHAN-CUSTOMER")

{'rule': 'T2-ORPHAN-CUSTOMER', 'method': 'supervised_ground_truth', 'reason': 'The invalid FK exposes the defect but not the original customer; restore the supervised relationship.', 'before': 1000, 'updated': 1000, 'after': 0}
After cleaning examples:
  {'primary_key': {'OrderID': 10293}, 'cleaned_values': {'CustomerID': 'TORTU'}, 'expected_clean': {'CustomerID': 'TORTU'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10301}, 'cleaned_values': {'CustomerID': 'WANDK'}, 'expected_clean': {'CustomerID': 'WANDK'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10313}, 'cleaned_values': {'CustomerID': 'QUICK'}, 'expected_clean': {'CustomerID': 'QUICK'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10324}, 'cleaned_values': {'CustomerID': 'SAVEA'}, 'expected_clean': {'CustomerID': 'SAVEA'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10345}, 'cleaned_values': {'CustomerID': 'QUICK'}, 'expected_clean': {'CustomerID': 'QUICK'}, 'matche

## Problem 5: Order tham chiếu EmployeeID không tồn tại

**Vị trí:** `Orders.EmployeeID`  
**Loại lỗi:** `foreign_key_violation`  
**Ground truth:** 600 assertions trên 600 rows

**Problem.** Orders.EmployeeID chứa khóa không tồn tại trong Employees. Giá trị sai vẫn đúng kiểu integer.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [12]:
# Detect Problem 5
problem_5_candidates = show_problem_examples("T2-ORPHAN-EMPLOYEE")

Candidate rows/groups: 600
Exact assertions: 600
Exact affected rows: 600
Verified raw examples:
  {'primary_key': {'OrderID': 10258}, 'raw_values': {'EmployeeID': 9999}}
  {'primary_key': {'OrderID': 10295}, 'raw_values': {'EmployeeID': 9999}}
  {'primary_key': {'OrderID': 10356}, 'raw_values': {'EmployeeID': 9999}}
  {'primary_key': {'OrderID': 10409}, 'raw_values': {'EmployeeID': 9999}}
  {'primary_key': {'OrderID': 10509}, 'raw_values': {'EmployeeID': 9999}}


### Solution 5

Dùng anti-join để phát hiện và supervised label để khôi phục employee assignment chính xác.

**Solution mode:** `reference_repair`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [13]:
def clean_orphan_employee():
    """Restore exact labels because raw candidates do not identify one unique clean value."""
    rule_id = "T2-ORPHAN-EMPLOYEE"
    before = unresolved_count(rule_id)
    updated = 0

    # Each label supplies table, column, clean value, and row identity.
    for record in records_for(rule_id):
        # A corrupted primary key must locate the raw row before restoring the key.
        locator = current_locator(record)
        where, parameters = primary_key_filter(locator)
        update_sql = (
            f"UPDATE {quote_name(record['table'])} "
            f"SET {quote_name(record['column'])} = ? WHERE {where}"
        )
        cursor = connection.execute(
            update_sql, [record["clean_value"], *parameters]
        )
        assert cursor.rowcount == 1, (rule_id, locator)
        updated += 1

    report = {
        "rule": rule_id,
        "method": "supervised_ground_truth",
        "reason": 'Restore the original employee relationship from supervised labels.',
        "before": before,
        "updated": updated,
        "after": unresolved_count(rule_id),
    }
    print(report)
    return report

problem_5_report = clean_orphan_employee()
show_repaired_examples("T2-ORPHAN-EMPLOYEE")

{'rule': 'T2-ORPHAN-EMPLOYEE', 'method': 'supervised_ground_truth', 'reason': 'Restore the original employee relationship from supervised labels.', 'before': 600, 'updated': 600, 'after': 0}
After cleaning examples:
  {'primary_key': {'OrderID': 10258}, 'cleaned_values': {'EmployeeID': 1}, 'expected_clean': {'EmployeeID': 1}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10295}, 'cleaned_values': {'EmployeeID': 2}, 'expected_clean': {'EmployeeID': 2}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10356}, 'cleaned_values': {'EmployeeID': 6}, 'expected_clean': {'EmployeeID': 6}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10409}, 'cleaned_values': {'EmployeeID': 3}, 'expected_clean': {'EmployeeID': 3}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10509}, 'cleaned_values': {'EmployeeID': 4}, 'expected_clean': {'EmployeeID': 4}, 'matches_ground_truth': True}


## Problem 6: Order tham chiếu Shipper không tồn tại

**Vị trí:** `Orders.ShipVia`  
**Loại lỗi:** `foreign_key_violation`  
**Ground truth:** 600 assertions trên 600 rows

**Problem.** Orders.ShipVia trỏ tới ShipperID không có trong Shippers và làm hỏng tính toàn vẹn quan hệ.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [14]:
# Detect Problem 6
problem_6_candidates = show_problem_examples("T2-ORPHAN-SHIPPER")

Candidate rows/groups: 600
Exact assertions: 600
Exact affected rows: 600
Verified raw examples:
  {'primary_key': {'OrderID': 10250}, 'raw_values': {'ShipVia': 9999}}
  {'primary_key': {'OrderID': 10254}, 'raw_values': {'ShipVia': 9999}}
  {'primary_key': {'OrderID': 10286}, 'raw_values': {'ShipVia': 9999}}
  {'primary_key': {'OrderID': 10312}, 'raw_values': {'ShipVia': 9999}}
  {'primary_key': {'OrderID': 10340}, 'raw_values': {'ShipVia': 9999}}


### Solution 6

Phát hiện bằng anti-join, sau đó phục hồi ShipVia gốc từ ground truth.

**Solution mode:** `reference_repair`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [15]:
def clean_orphan_shipper():
    """Restore exact labels because raw candidates do not identify one unique clean value."""
    rule_id = "T2-ORPHAN-SHIPPER"
    before = unresolved_count(rule_id)
    updated = 0

    # Each label supplies table, column, clean value, and row identity.
    for record in records_for(rule_id):
        # A corrupted primary key must locate the raw row before restoring the key.
        locator = current_locator(record)
        where, parameters = primary_key_filter(locator)
        update_sql = (
            f"UPDATE {quote_name(record['table'])} "
            f"SET {quote_name(record['column'])} = ? WHERE {where}"
        )
        cursor = connection.execute(
            update_sql, [record["clean_value"], *parameters]
        )
        assert cursor.rowcount == 1, (rule_id, locator)
        updated += 1

    report = {
        "rule": rule_id,
        "method": "supervised_ground_truth",
        "reason": 'Restore the original shipper relationship from supervised labels.',
        "before": before,
        "updated": updated,
        "after": unresolved_count(rule_id),
    }
    print(report)
    return report

problem_6_report = clean_orphan_shipper()
show_repaired_examples("T2-ORPHAN-SHIPPER")

{'rule': 'T2-ORPHAN-SHIPPER', 'method': 'supervised_ground_truth', 'reason': 'Restore the original shipper relationship from supervised labels.', 'before': 600, 'updated': 600, 'after': 0}
After cleaning examples:
  {'primary_key': {'OrderID': 10250}, 'cleaned_values': {'ShipVia': 2}, 'expected_clean': {'ShipVia': 2}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10254}, 'cleaned_values': {'ShipVia': 2}, 'expected_clean': {'ShipVia': 2}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10286}, 'cleaned_values': {'ShipVia': 3}, 'expected_clean': {'ShipVia': 3}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10312}, 'cleaned_values': {'ShipVia': 2}, 'expected_clean': {'ShipVia': 2}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10340}, 'cleaned_values': {'ShipVia': 3}, 'expected_clean': {'ShipVia': 3}, 'matches_ground_truth': True}


## Problem 7: Order detail tham chiếu ProductID không tồn tại

**Vị trí:** `Order Details.ProductID`  
**Loại lỗi:** `foreign_key_violation`  
**Ground truth:** 800 assertions trên 800 rows

**Problem.** ProductID là một phần composite primary key của Order Details và đã bị đổi sang khóa sản phẩm không tồn tại.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [16]:
# Detect Problem 7
problem_7_candidates = show_problem_examples("T2-ORPHAN-DETAIL-PRODUCT")

Candidate rows/groups: 800
Exact assertions: 800
Exact affected rows: 800
Verified raw examples:
  {'primary_key': {'OrderID': 10577, 'ProductID': 39}, 'raw_values': {'ProductID': 100039}}
  {'primary_key': {'OrderID': 10683, 'ProductID': 52}, 'raw_values': {'ProductID': 100052}}
  {'primary_key': {'OrderID': 11012, 'ProductID': 60}, 'raw_values': {'ProductID': 100060}}
  {'primary_key': {'OrderID': 11111, 'ProductID': 74}, 'raw_values': {'ProductID': 100074}}
  {'primary_key': {'OrderID': 11151, 'ProductID': 55}, 'raw_values': {'ProductID': 100055}}


### Solution 7

Dùng corrupted_primary_key để định vị raw row rồi phục hồi ProductID gốc từ supervised label.

**Solution mode:** `reference_repair`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [17]:
def clean_orphan_detail_product():
    """Restore exact labels because raw candidates do not identify one unique clean value."""
    rule_id = "T2-ORPHAN-DETAIL-PRODUCT"
    before = unresolved_count(rule_id)
    updated = 0

    # Each label supplies table, column, clean value, and row identity.
    for record in records_for(rule_id):
        # A corrupted primary key must locate the raw row before restoring the key.
        locator = current_locator(record)
        where, parameters = primary_key_filter(locator)
        update_sql = (
            f"UPDATE {quote_name(record['table'])} "
            f"SET {quote_name(record['column'])} = ? WHERE {where}"
        )
        cursor = connection.execute(
            update_sql, [record["clean_value"], *parameters]
        )
        assert cursor.rowcount == 1, (rule_id, locator)
        updated += 1

    report = {
        "rule": rule_id,
        "method": "supervised_ground_truth",
        "reason": 'Restore the original composite-key ProductID from supervised labels.',
        "before": before,
        "updated": updated,
        "after": unresolved_count(rule_id),
    }
    print(report)
    return report

problem_7_report = clean_orphan_detail_product()
show_repaired_examples("T2-ORPHAN-DETAIL-PRODUCT")

{'rule': 'T2-ORPHAN-DETAIL-PRODUCT', 'method': 'supervised_ground_truth', 'reason': 'Restore the original composite-key ProductID from supervised labels.', 'before': 800, 'updated': 800, 'after': 0}
After cleaning examples:
  {'primary_key': {'OrderID': 10577, 'ProductID': 39}, 'cleaned_values': {'ProductID': 39}, 'expected_clean': {'ProductID': 39}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10683, 'ProductID': 52}, 'cleaned_values': {'ProductID': 52}, 'expected_clean': {'ProductID': 52}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 11012, 'ProductID': 60}, 'cleaned_values': {'ProductID': 60}, 'expected_clean': {'ProductID': 60}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 11111, 'ProductID': 74}, 'cleaned_values': {'ProductID': 74}, 'expected_clean': {'ProductID': 74}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 11151, 'ProductID': 55}, 'cleaned_values': {'ProductID': 55}, 'expected_clean': {'ProductID': 55}, 'matches_gr

## Problem 8: Product tham chiếu Supplier không tồn tại

**Vị trí:** `Products.SupplierID`  
**Loại lỗi:** `foreign_key_violation`  
**Ground truth:** 25 assertions trên 25 rows

**Problem.** Products.SupplierID không có bản ghi cha tương ứng trong Suppliers.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [18]:
# Detect Problem 8
problem_8_candidates = show_problem_examples("T2-ORPHAN-PRODUCT-SUPPLIER")

Candidate rows/groups: 25
Exact assertions: 25
Exact affected rows: 25
Verified raw examples:
  {'primary_key': {'ProductID': 11}, 'raw_values': {'SupplierID': 9999}}
  {'primary_key': {'ProductID': 12}, 'raw_values': {'SupplierID': 9999}}
  {'primary_key': {'ProductID': 13}, 'raw_values': {'SupplierID': 9999}}
  {'primary_key': {'ProductID': 14}, 'raw_values': {'SupplierID': 9999}}
  {'primary_key': {'ProductID': 16}, 'raw_values': {'SupplierID': 9999}}


### Solution 8

Phát hiện bằng anti-join và phục hồi supplier relationship từ ground truth.

**Solution mode:** `reference_repair`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [19]:
def clean_orphan_product_supplier():
    """Restore exact labels because raw candidates do not identify one unique clean value."""
    rule_id = "T2-ORPHAN-PRODUCT-SUPPLIER"
    before = unresolved_count(rule_id)
    updated = 0

    # Each label supplies table, column, clean value, and row identity.
    for record in records_for(rule_id):
        # A corrupted primary key must locate the raw row before restoring the key.
        locator = current_locator(record)
        where, parameters = primary_key_filter(locator)
        update_sql = (
            f"UPDATE {quote_name(record['table'])} "
            f"SET {quote_name(record['column'])} = ? WHERE {where}"
        )
        cursor = connection.execute(
            update_sql, [record["clean_value"], *parameters]
        )
        assert cursor.rowcount == 1, (rule_id, locator)
        updated += 1

    report = {
        "rule": rule_id,
        "method": "supervised_ground_truth",
        "reason": 'Restore the original Product-to-Supplier relationship.',
        "before": before,
        "updated": updated,
        "after": unresolved_count(rule_id),
    }
    print(report)
    return report

problem_8_report = clean_orphan_product_supplier()
show_repaired_examples("T2-ORPHAN-PRODUCT-SUPPLIER")

{'rule': 'T2-ORPHAN-PRODUCT-SUPPLIER', 'method': 'supervised_ground_truth', 'reason': 'Restore the original Product-to-Supplier relationship.', 'before': 25, 'updated': 25, 'after': 0}
After cleaning examples:
  {'primary_key': {'ProductID': 11}, 'cleaned_values': {'SupplierID': 5}, 'expected_clean': {'SupplierID': 5}, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 12}, 'cleaned_values': {'SupplierID': 5}, 'expected_clean': {'SupplierID': 5}, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 13}, 'cleaned_values': {'SupplierID': 6}, 'expected_clean': {'SupplierID': 6}, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 14}, 'cleaned_values': {'SupplierID': 6}, 'expected_clean': {'SupplierID': 6}, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 16}, 'cleaned_values': {'SupplierID': 7}, 'expected_clean': {'SupplierID': 7}, 'matches_ground_truth': True}


## Problem 9: Product tham chiếu Category không tồn tại

**Vị trí:** `Products.CategoryID`  
**Loại lỗi:** `foreign_key_violation`  
**Ground truth:** 25 assertions trên 25 rows

**Problem.** Products.CategoryID không có bản ghi cha tương ứng trong Categories.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [20]:
# Detect Problem 9
problem_9_candidates = show_problem_examples("T2-ORPHAN-PRODUCT-CATEGORY")

Candidate rows/groups: 25
Exact assertions: 25
Exact affected rows: 25
Verified raw examples:
  {'primary_key': {'ProductID': 11}, 'raw_values': {'CategoryID': 9999}}
  {'primary_key': {'ProductID': 13}, 'raw_values': {'CategoryID': 9999}}
  {'primary_key': {'ProductID': 14}, 'raw_values': {'CategoryID': 9999}}
  {'primary_key': {'ProductID': 17}, 'raw_values': {'CategoryID': 9999}}
  {'primary_key': {'ProductID': 18}, 'raw_values': {'CategoryID': 9999}}


### Solution 9

Phát hiện bằng anti-join và phục hồi category relationship từ ground truth.

**Solution mode:** `reference_repair`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [21]:
def clean_orphan_product_category():
    """Restore exact labels because raw candidates do not identify one unique clean value."""
    rule_id = "T2-ORPHAN-PRODUCT-CATEGORY"
    before = unresolved_count(rule_id)
    updated = 0

    # Each label supplies table, column, clean value, and row identity.
    for record in records_for(rule_id):
        # A corrupted primary key must locate the raw row before restoring the key.
        locator = current_locator(record)
        where, parameters = primary_key_filter(locator)
        update_sql = (
            f"UPDATE {quote_name(record['table'])} "
            f"SET {quote_name(record['column'])} = ? WHERE {where}"
        )
        cursor = connection.execute(
            update_sql, [record["clean_value"], *parameters]
        )
        assert cursor.rowcount == 1, (rule_id, locator)
        updated += 1

    report = {
        "rule": rule_id,
        "method": "supervised_ground_truth",
        "reason": 'Restore the original Product-to-Category relationship.',
        "before": before,
        "updated": updated,
        "after": unresolved_count(rule_id),
    }
    print(report)
    return report

problem_9_report = clean_orphan_product_category()
show_repaired_examples("T2-ORPHAN-PRODUCT-CATEGORY")

{'rule': 'T2-ORPHAN-PRODUCT-CATEGORY', 'method': 'supervised_ground_truth', 'reason': 'Restore the original Product-to-Category relationship.', 'before': 25, 'updated': 25, 'after': 0}
After cleaning examples:
  {'primary_key': {'ProductID': 11}, 'cleaned_values': {'CategoryID': 4}, 'expected_clean': {'CategoryID': 4}, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 13}, 'cleaned_values': {'CategoryID': 8}, 'expected_clean': {'CategoryID': 8}, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 14}, 'cleaned_values': {'CategoryID': 7}, 'expected_clean': {'CategoryID': 7}, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 17}, 'cleaned_values': {'CategoryID': 6}, 'expected_clean': {'CategoryID': 6}, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 18}, 'cleaned_values': {'CategoryID': 8}, 'expected_clean': {'CategoryID': 8}, 'matches_ground_truth': True}


## Problem 10: RequiredDate xảy ra trước OrderDate

**Vị trí:** `Orders.RequiredDate`  
**Loại lỗi:** `date_business_rule`  
**Ground truth:** 900 assertions trên 900 rows

**Problem.** Ngày yêu cầu giao hàng sớm hơn ngày đặt hàng, vi phạm business rule dù cả hai ngày đều parse hợp lệ.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [22]:
# Detect Problem 10
problem_10_candidates = show_problem_examples("T2-REQUIRED-BEFORE-ORDER")

Candidate rows/groups: 900
Exact assertions: 900
Exact affected rows: 900
Verified raw examples:
  {'primary_key': {'OrderID': 10273}, 'raw_values': {'RequiredDate': '2016-07-06 00:00:00'}}
  {'primary_key': {'OrderID': 10285}, 'raw_values': {'RequiredDate': '2016-07-21 00:00:00'}}
  {'primary_key': {'OrderID': 10327}, 'raw_values': {'RequiredDate': '2016-09-11 00:00:00'}}
  {'primary_key': {'OrderID': 10361}, 'raw_values': {'RequiredDate': '2016-10-23 00:00:00'}}
  {'primary_key': {'OrderID': 10388}, 'raw_values': {'RequiredDate': '2016-11-19 00:00:00'}}


### Solution 10

Rule thời gian phát hiện lỗi nhưng không suy ra được ngày hẹn gốc; dùng supervised date để phục hồi chính xác.

**Solution mode:** `oracle_required`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [23]:
def clean_required_before_order():
    """Restore exact labels because raw candidates do not identify one unique clean value."""
    rule_id = "T2-REQUIRED-BEFORE-ORDER"
    before = unresolved_count(rule_id)
    updated = 0

    # Each label supplies table, column, clean value, and row identity.
    for record in records_for(rule_id):
        # A corrupted primary key must locate the raw row before restoring the key.
        locator = current_locator(record)
        where, parameters = primary_key_filter(locator)
        update_sql = (
            f"UPDATE {quote_name(record['table'])} "
            f"SET {quote_name(record['column'])} = ? WHERE {where}"
        )
        cursor = connection.execute(
            update_sql, [record["clean_value"], *parameters]
        )
        assert cursor.rowcount == 1, (rule_id, locator)
        updated += 1

    report = {
        "rule": rule_id,
        "method": "supervised_ground_truth",
        "reason": 'The date rule detects the defect but cannot infer the exact promised date; restore the supervised date.',
        "before": before,
        "updated": updated,
        "after": unresolved_count(rule_id),
    }
    print(report)
    return report

problem_10_report = clean_required_before_order()
show_repaired_examples("T2-REQUIRED-BEFORE-ORDER")

{'rule': 'T2-REQUIRED-BEFORE-ORDER', 'method': 'supervised_ground_truth', 'reason': 'The date rule detects the defect but cannot infer the exact promised date; restore the supervised date.', 'before': 900, 'updated': 900, 'after': 0}
After cleaning examples:
  {'primary_key': {'OrderID': 10273}, 'cleaned_values': {'RequiredDate': '2016-09-02'}, 'expected_clean': {'RequiredDate': '2016-09-02'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10285}, 'cleaned_values': {'RequiredDate': '2016-09-17'}, 'expected_clean': {'RequiredDate': '2016-09-17'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10327}, 'cleaned_values': {'RequiredDate': '2016-11-08'}, 'expected_clean': {'RequiredDate': '2016-11-08'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10361}, 'cleaned_values': {'RequiredDate': '2016-12-20'}, 'expected_clean': {'RequiredDate': '2016-12-20'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10388}, 'cleaned_values': {'RequiredDat

## Problem 11: ShippedDate xảy ra trước OrderDate

**Vị trí:** `Orders.ShippedDate`  
**Loại lỗi:** `date_business_rule`  
**Ground truth:** 900 assertions trên 900 rows

**Problem.** Ngày giao hàng sớm hơn ngày đặt, tạo trình tự thời gian không thể xảy ra.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [24]:
# Detect Problem 11
problem_11_candidates = show_problem_examples("T2-SHIPPED-BEFORE-ORDER")

Candidate rows/groups: 900
Exact assertions: 900
Exact affected rows: 900
Verified raw examples:
  {'primary_key': {'OrderID': 10254}, 'raw_values': {'ShippedDate': '2016-07-01 00:00:00'}}
  {'primary_key': {'OrderID': 10295}, 'raw_values': {'ShippedDate': '2016-08-23 00:00:00'}}
  {'primary_key': {'OrderID': 10297}, 'raw_values': {'ShippedDate': '2016-08-25 00:00:00'}}
  {'primary_key': {'OrderID': 10329}, 'raw_values': {'ShippedDate': '2016-10-05 00:00:00'}}
  {'primary_key': {'OrderID': 10339}, 'raw_values': {'ShippedDate': '2016-10-18 00:00:00'}}


### Solution 11

Dùng comparison giữa hai timestamp để phát hiện và supervised label để phục hồi ngày giao thực tế.

**Solution mode:** `oracle_required`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [25]:
def clean_shipped_before_order():
    """Restore exact labels because raw candidates do not identify one unique clean value."""
    rule_id = "T2-SHIPPED-BEFORE-ORDER"
    before = unresolved_count(rule_id)
    updated = 0

    # Each label supplies table, column, clean value, and row identity.
    for record in records_for(rule_id):
        # A corrupted primary key must locate the raw row before restoring the key.
        locator = current_locator(record)
        where, parameters = primary_key_filter(locator)
        update_sql = (
            f"UPDATE {quote_name(record['table'])} "
            f"SET {quote_name(record['column'])} = ? WHERE {where}"
        )
        cursor = connection.execute(
            update_sql, [record["clean_value"], *parameters]
        )
        assert cursor.rowcount == 1, (rule_id, locator)
        updated += 1

    report = {
        "rule": rule_id,
        "method": "supervised_ground_truth",
        "reason": 'The date rule detects the defect but cannot infer the exact shipped date; restore the supervised date.',
        "before": before,
        "updated": updated,
        "after": unresolved_count(rule_id),
    }
    print(report)
    return report

problem_11_report = clean_shipped_before_order()
show_repaired_examples("T2-SHIPPED-BEFORE-ORDER")

{'rule': 'T2-SHIPPED-BEFORE-ORDER', 'method': 'supervised_ground_truth', 'reason': 'The date rule detects the defect but cannot infer the exact shipped date; restore the supervised date.', 'before': 900, 'updated': 900, 'after': 0}
After cleaning examples:
  {'primary_key': {'OrderID': 10254}, 'cleaned_values': {'ShippedDate': '2016-07-23'}, 'expected_clean': {'ShippedDate': '2016-07-23'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10295}, 'cleaned_values': {'ShippedDate': '2016-09-10'}, 'expected_clean': {'ShippedDate': '2016-09-10'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10297}, 'cleaned_values': {'ShippedDate': '2016-09-10'}, 'expected_clean': {'ShippedDate': '2016-09-10'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10329}, 'cleaned_values': {'ShippedDate': '2016-10-23'}, 'expected_clean': {'ShippedDate': '2016-10-23'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10339}, 'cleaned_values': {'ShippedDate': '2016-1

## Problem 12: Freight có giá trị âm

**Vị trí:** `Orders.Freight`  
**Loại lỗi:** `invalid_numeric_domain`  
**Ground truth:** 800 assertions trên 800 rows

**Problem.** Chi phí vận chuyển âm vi phạm miền nghiệp vụ. Corruption đã đổi dấu giá trị gốc.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [26]:
# Detect Problem 12
problem_12_candidates = show_problem_examples("T2-NEGATIVE-FREIGHT")

Candidate rows/groups: 800
Exact assertions: 800
Exact affected rows: 800
Verified raw examples:
  {'primary_key': {'OrderID': 10291}, 'raw_values': {'Freight': -21.5}}
  {'primary_key': {'OrderID': 10293}, 'raw_values': {'Freight': -18.25}}
  {'primary_key': {'OrderID': 10303}, 'raw_values': {'Freight': -31.25}}
  {'primary_key': {'OrderID': 10315}, 'raw_values': {'Freight': -21}}
  {'primary_key': {'OrderID': 10335}, 'raw_values': {'Freight': -31.5}}


### Solution 12

Xác nhận Freight nhỏ hơn 0 rồi lấy trị tuyệt đối; phép sửa được suy ra trực tiếp từ raw.

**Solution mode:** `deterministic_transform`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [27]:
def clean_negative_freight():
    """Undo the injected sign change while preserving integer/float representation."""
    rule_id = "T2-NEGATIVE-FREIGHT"

    def transform_raw_value(value, record):
        number = abs(value)
        return int(number) if isinstance(number, float) and number.is_integer() else number

    # apply_transform reads current raw values, calculates repaired values,
    # checks every result against clean labels, and performs parameterized UPDATEs.
    return apply_transform(rule_id, transform_raw_value)

problem_12_report = clean_negative_freight()
show_repaired_examples("T2-NEGATIVE-FREIGHT")

{'rule': 'T2-NEGATIVE-FREIGHT', 'before': 800, 'updated': 800, 'after': 0}
After cleaning examples:
  {'primary_key': {'OrderID': 10291}, 'cleaned_values': {'Freight': 21.5}, 'expected_clean': {'Freight': 21.5}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10293}, 'cleaned_values': {'Freight': 18.25}, 'expected_clean': {'Freight': 18.25}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10303}, 'cleaned_values': {'Freight': 31.25}, 'expected_clean': {'Freight': 31.25}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10315}, 'cleaned_values': {'Freight': 21}, 'expected_clean': {'Freight': 21}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10335}, 'cleaned_values': {'Freight': 31.5}, 'expected_clean': {'Freight': 31.5}, 'matches_ground_truth': True}


## Problem 13: Order quantity không dương

**Vị trí:** `Order Details.Quantity`  
**Loại lỗi:** `invalid_numeric_domain`  
**Ground truth:** 800 assertions trên 800 rows

**Problem.** Quantity bằng hoặc nhỏ hơn 0 là không hợp lệ đối với một order line trong bộ dữ liệu này.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [28]:
# Detect Problem 13
problem_13_candidates = show_problem_examples("T2-NEGATIVE-QUANTITY")

Candidate rows/groups: 800
Exact assertions: 800
Exact affected rows: 800
Verified raw examples:
  {'primary_key': {'OrderID': 10541, 'ProductID': 38}, 'raw_values': {'Quantity': -4}}
  {'primary_key': {'OrderID': 10833, 'ProductID': 53}, 'raw_values': {'Quantity': -9}}
  {'primary_key': {'OrderID': 10999, 'ProductID': 51}, 'raw_values': {'Quantity': -15}}
  {'primary_key': {'OrderID': 11024, 'ProductID': 33}, 'raw_values': {'Quantity': -30}}
  {'primary_key': {'OrderID': 11084, 'ProductID': 35}, 'raw_values': {'Quantity': -41}}


### Solution 13

Lấy trị tuyệt đối nguyên của đúng affected rows và đối chiếu clean labels.

**Solution mode:** `deterministic_transform`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [29]:
def clean_negative_quantity():
    """Undo the injected sign change and keep Quantity as an integer."""
    rule_id = "T2-NEGATIVE-QUANTITY"

    def transform_raw_value(value, record):
        return abs(int(value))

    # apply_transform reads current raw values, calculates repaired values,
    # checks every result against clean labels, and performs parameterized UPDATEs.
    return apply_transform(rule_id, transform_raw_value)

problem_13_report = clean_negative_quantity()
show_repaired_examples("T2-NEGATIVE-QUANTITY")

{'rule': 'T2-NEGATIVE-QUANTITY', 'before': 800, 'updated': 800, 'after': 0}
After cleaning examples:
  {'primary_key': {'OrderID': 10541, 'ProductID': 38}, 'cleaned_values': {'Quantity': 4}, 'expected_clean': {'Quantity': 4}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10833, 'ProductID': 53}, 'cleaned_values': {'Quantity': 9}, 'expected_clean': {'Quantity': 9}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10999, 'ProductID': 51}, 'cleaned_values': {'Quantity': 15}, 'expected_clean': {'Quantity': 15}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 11024, 'ProductID': 33}, 'cleaned_values': {'Quantity': 30}, 'expected_clean': {'Quantity': 30}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 11084, 'ProductID': 35}, 'cleaned_values': {'Quantity': 41}, 'expected_clean': {'Quantity': 41}, 'matches_ground_truth': True}


## Problem 14: Discount nằm ngoài miền 0–1

**Vị trí:** `Order Details.Discount`  
**Loại lỗi:** `invalid_numeric_domain`  
**Ground truth:** 600 assertions trên 600 rows

**Problem.** Fractional discount bị lưu thành whole percentage như 15 thay vì 0.15.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [30]:
# Detect Problem 14
problem_14_candidates = show_problem_examples("T2-DISCOUNT-OUTSIDE-RANGE")

Candidate rows/groups: 600
Exact assertions: 600
Exact affected rows: 600
Verified raw examples:
  {'primary_key': {'OrderID': 10250, 'ProductID': 51}, 'raw_values': {'Discount': 15.0}}
  {'primary_key': {'OrderID': 10250, 'ProductID': 65}, 'raw_values': {'Discount': 15.0}}
  {'primary_key': {'OrderID': 10251, 'ProductID': 22}, 'raw_values': {'Discount': 5.0}}
  {'primary_key': {'OrderID': 10252, 'ProductID': 20}, 'raw_values': {'Discount': 5.0}}
  {'primary_key': {'OrderID': 10252, 'ProductID': 33}, 'raw_values': {'Discount': 5.0}}


### Solution 14

Chia injected value cho 100 rồi kiểm tra kết quả nằm trong miền hợp lệ.

**Solution mode:** `deterministic_transform`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [31]:
def clean_discount_outside_range():
    """Convert a whole percentage back to its fractional representation."""
    rule_id = "T2-DISCOUNT-OUTSIDE-RANGE"

    def transform_raw_value(value, record):
        return float(value) / 100

    # apply_transform reads current raw values, calculates repaired values,
    # checks every result against clean labels, and performs parameterized UPDATEs.
    return apply_transform(rule_id, transform_raw_value)

problem_14_report = clean_discount_outside_range()
show_repaired_examples("T2-DISCOUNT-OUTSIDE-RANGE")

{'rule': 'T2-DISCOUNT-OUTSIDE-RANGE', 'before': 600, 'updated': 600, 'after': 0}
After cleaning examples:
  {'primary_key': {'OrderID': 10250, 'ProductID': 51}, 'cleaned_values': {'Discount': 0.15}, 'expected_clean': {'Discount': 0.15}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10250, 'ProductID': 65}, 'cleaned_values': {'Discount': 0.15}, 'expected_clean': {'Discount': 0.15}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10251, 'ProductID': 22}, 'cleaned_values': {'Discount': 0.05}, 'expected_clean': {'Discount': 0.05}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10252, 'ProductID': 20}, 'cleaned_values': {'Discount': 0.05}, 'expected_clean': {'Discount': 0.05}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10252, 'ProductID': 33}, 'cleaned_values': {'Discount': 0.05}, 'expected_clean': {'Discount': 0.05}, 'matches_ground_truth': True}


## Problem 15: UnitsInStock có giá trị âm

**Vị trí:** `Products.UnitsInStock`  
**Loại lỗi:** `invalid_numeric_domain`  
**Ground truth:** 30 assertions trên 30 rows

**Problem.** Tồn kho âm vi phạm domain rule. Raw -1 có thể xuất phát từ clean 0 hoặc 1 nên trị tuyệt đối không luôn cho đáp án duy nhất.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [32]:
# Detect Problem 15
problem_15_candidates = show_problem_examples("T2-NEGATIVE-STOCK")

Candidate rows/groups: 30
Exact assertions: 30
Exact affected rows: 30
Verified raw examples:
  {'primary_key': {'ProductID': 15}, 'raw_values': {'UnitsInStock': -39}}
  {'primary_key': {'ProductID': 18}, 'raw_values': {'UnitsInStock': -42}}
  {'primary_key': {'ProductID': 19}, 'raw_values': {'UnitsInStock': -25}}
  {'primary_key': {'ProductID': 1}, 'raw_values': {'UnitsInStock': -39}}
  {'primary_key': {'ProductID': 28}, 'raw_values': {'UnitsInStock': -26}}


### Solution 15

Phát hiện bằng điều kiện nhỏ hơn 0 và dùng supervised labels cho exact restoration.

**Solution mode:** `oracle_required`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [33]:
def clean_negative_stock():
    """Restore exact labels because raw candidates do not identify one unique clean value."""
    rule_id = "T2-NEGATIVE-STOCK"
    before = unresolved_count(rule_id)
    updated = 0

    # Each label supplies table, column, clean value, and row identity.
    for record in records_for(rule_id):
        # A corrupted primary key must locate the raw row before restoring the key.
        locator = current_locator(record)
        where, parameters = primary_key_filter(locator)
        update_sql = (
            f"UPDATE {quote_name(record['table'])} "
            f"SET {quote_name(record['column'])} = ? WHERE {where}"
        )
        cursor = connection.execute(
            update_sql, [record["clean_value"], *parameters]
        )
        assert cursor.rowcount == 1, (rule_id, locator)
        updated += 1

    report = {
        "rule": rule_id,
        "method": "supervised_ground_truth",
        "reason": 'Most values are negated, but raw -1 is ambiguous between clean 0 and 1; exact restoration uses labels.',
        "before": before,
        "updated": updated,
        "after": unresolved_count(rule_id),
    }
    print(report)
    return report

problem_15_report = clean_negative_stock()
show_repaired_examples("T2-NEGATIVE-STOCK")

{'rule': 'T2-NEGATIVE-STOCK', 'method': 'supervised_ground_truth', 'reason': 'Most values are negated, but raw -1 is ambiguous between clean 0 and 1; exact restoration uses labels.', 'before': 30, 'updated': 30, 'after': 0}
After cleaning examples:
  {'primary_key': {'ProductID': 15}, 'cleaned_values': {'UnitsInStock': 39}, 'expected_clean': {'UnitsInStock': 39}, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 18}, 'cleaned_values': {'UnitsInStock': 42}, 'expected_clean': {'UnitsInStock': 42}, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 19}, 'cleaned_values': {'UnitsInStock': 25}, 'expected_clean': {'UnitsInStock': 25}, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 1}, 'cleaned_values': {'UnitsInStock': 39}, 'expected_clean': {'UnitsInStock': 39}, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 28}, 'cleaned_values': {'UnitsInStock': 26}, 'expected_clean': {'UnitsInStock': 26}, 'matches_ground_truth': True}


## Final validation

Tất cả faults phải được repair, clean controls không được thay đổi, foreign keys phải hợp lệ và 13 table fingerprints phải khớp expected clean state trong JSON.

In [34]:
def primary_key_columns(table):
    rows = connection.execute(f"PRAGMA table_info({quote_name(table)})").fetchall()
    return [row["name"] for row in sorted(
        (row for row in rows if row["pk"]), key=lambda row: row["pk"])]

def table_fingerprint(table):
    keys = primary_key_columns(table)
    query = f"SELECT * FROM {quote_name(table)}"
    if keys:
        query += " ORDER BY " + ", ".join(quote_name(key) for key in keys)
    digest = hashlib.sha256()
    for row in connection.execute(query):
        values = [{"blob_sha256": hashlib.sha256(value).hexdigest(), "bytes": len(value)}
                  if isinstance(value, bytes) else value for value in row]
        line = json.dumps(values, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
        digest.update(line.encode("utf-8") + b"\n")
    return digest.hexdigest()

def score_ground_truth():
    repaired = remaining = preserved = false_positive = 0
    for record in bundle["repair_records"]:
        row = find_row(record["table"], record["primary_key"])
        if record["operation"] == "insert_row":
            repaired += int(row is None); remaining += int(row is not None)
        elif row is not None and row[record["column"]] == record["clean_value"]:
            repaired += 1
        else:
            remaining += 1
    for control in bundle["clean_controls"]:
        row = find_row(control["table"], control["primary_key"])
        if row is not None and row[control["column"]] == control["clean_value"]:
            preserved += 1
        else:
            false_positive += 1
    total = repaired + remaining + preserved + false_positive
    return {"quality_score": round(100 * (repaired + preserved) / total, 4),
            "repaired_faults": repaired, "remaining_faults": remaining,
            "preserved_clean_controls": preserved,
            "false_positive_controls": false_positive}

connection.execute("DELETE FROM sqlite_sequence")
connection.executemany("INSERT INTO sqlite_sequence(name, seq) VALUES (?, ?)",
    [(item["name"], item["seq"])
     for item in bundle["validation_expectations"]["sqlite_sequence"]])
connection.commit()

expected = bundle["validation_expectations"]
matching_tables = 0
for table, wanted in expected["tables"].items():
    rows = connection.execute(f"SELECT COUNT(*) FROM {quote_name(table)}").fetchone()[0]
    matching_tables += int(rows == wanted["rows"]
                           and table_fingerprint(table) == wanted["fingerprint"])
sequence = [dict(row) for row in connection.execute(
    "SELECT name, seq FROM sqlite_sequence ORDER BY name")]
final_validation = {
    **score_ground_truth(),
    "integrity_check": connection.execute("PRAGMA integrity_check").fetchone()[0],
    "foreign_key_violations": len(connection.execute("PRAGMA foreign_key_check").fetchall()),
    "matching_tables": matching_tables,
    "table_count": len(expected["tables"]),
    "sqlite_sequence_match": sequence == expected["sqlite_sequence"],
}
final_validation["passed"] = (
    final_validation["quality_score"] == 100.0
    and final_validation["remaining_faults"] == 0
    and final_validation["false_positive_controls"] == 0
    and final_validation["integrity_check"] == "ok"
    and final_validation["foreign_key_violations"] == 0
    and matching_tables == len(expected["tables"])
    and final_validation["sqlite_sequence_match"])
print(json.dumps(final_validation, indent=2))
assert final_validation["passed"]

{
  "quality_score": 100.0,
  "repaired_faults": 8580,
  "remaining_faults": 0,
  "preserved_clean_controls": 7020,
  "false_positive_controls": 0,
  "integrity_check": "ok",
  "foreign_key_violations": 0,
  "matching_tables": 13,
  "table_count": 13,
  "sqlite_sequence_match": true,
  "passed": true
}


## Publish `northwind_cleaned2.db`

Chỉ sau khi validation pass, working database mới được ghi ra clean output. Raw database không bị thay đổi.

In [35]:
descriptor, temporary_name = tempfile.mkstemp(
    prefix=".notebook_clean_", suffix=".db", dir=FOLDER)
os.close(descriptor)
temporary_path = Path(temporary_name)
try:
    destination = sqlite3.connect(temporary_path)
    connection.backup(destination)
    destination.close()
    os.replace(temporary_path, CLEAN)
finally:
    temporary_path.unlink(missing_ok=True)
print("Published:", CLEAN.name)
print("Output SHA-256:", sha256_file(CLEAN))
print("Raw remained unchanged:", sha256_file(RAW) == bundle["dataset"]["raw_sha256"])
connection.close()

Published: northwind_cleaned2.db
Output SHA-256: 60abef0cd9b0e701aaa08490d522a206cc6acf43707040184e821f9a4a601c5c
Raw remained unchanged: True


## Hoàn thành

JSON là machine-readable knowledge, notebook là tài liệu problem/solution trực quan, và Python là executable reference answer end-to-end.